* A convolution is not only defined by its kernel.
* Padding decides how the border is treated, and stride decides how densely the kernel samples locations.

* These choices control spatial resolution, information loss, computation, and the shape contracts between layers.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain padding and stride as architectural choices, not only formula inputs
- compute convolution output sizes as ordinary Python code
- verify padding and stride against `nn.Conv2d`
- explain why padding preserves border information
- debug impossible kernel/input geometry

In [1]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def conv_out_size(input_size, kernel_size, padding=0, stride=1):
    return (input_size + 2 * padding - kernel_size) // stride + 1

# 7.3.0 The Problem This Notebook Solves

Chapter 7.2 used valid convolution: the kernel only visited positions where it fit fully inside the input.

That made the output smaller. If we stack many such layers, spatial maps can shrink quickly.

Padding and stride are the two main controls introduced here:

- **Padding adds border values before the kernel slides**.
- **Stride changes how far the kernel moves between windows**.

These are not cosmetic settings. They change the representation pipeline:

```text
padding controls border treatment and spatial preservation
stride controls downsampling and compute
kernel size controls local receptive field
```

The key habit is to predict output shape before connecting layers.
* Shape formulas are not abstract math here; they are engineering contracts.
* If one layer produces an unexpected height or width, the next layer may fail or silently receive a different representation.

# 7.3.1 Output Size Is a Mechanical Contract

For one spatial dimension, use this code pattern:

```text
add padding on both sides
subtract the kernel size
count stride steps that fit
include the first position
```

The formula is written as ordinary Python because the purpose is shape reasoning, not symbolic manipulation:

```python
out = (input_size + 2 * padding - kernel_size) // stride + 1
```

The integer division matters.
* With stride > 1, not every possible window start is visited.
* e.g. with `stride = 2`, the kernel jumps two positions at a time:
> * `[0, 0, 1, 1]` are only inspected as `[0, 0]` and `[1, 1]`, not `[0, 0]`, `[0, 1]` and `[1, 1]` if stride = 1
* The output counts only starts where the kernel still fits.

In [ ]:
configs = [
    {"input_size": 8, "kernel_size": 3, "padding":0, "stride": 1},
    {"input_size": 8, "kernel_size": 3, "padding":1, "stride": 1},
    {"input_size": 8, "kernel_size": 3, "padding":0, "stride": 2},
]

for cfg in configs:
    out = conv_out_size(**cfg) # Calculate the expected spatial output size for this config
    conv = nn.Conv2d(1, 1, cfg["kernel_size"], padding=cfg["padding"], stride=cfg["stride"]) # Create a Conv2d layer using the current config
    Y = conv(torch.zeros(1, 1, cfg["input_size"], cfg["input_size"])) # Pass a zero input through the layer to get its output
    print(cfg, "formula:", out, "pytorch:", shape(Y))
    assert shape(Y)[2:] == (out, out) # (1, 1, 6, 6) since kernel_size = 3 has 6 possible positions, or use 8 - 3 + 1 = 6

# 7.3.2 Padding Adds Artificial Border Values

Without padding, border pixels participate in fewer windows than center pixels.

A corner pixel might appear in only one 3 by 3 valid window, while a center pixel appears in many. Padding changes that treatment.

Padding can preserve spatial size, which is useful when stacking layers because it prevents feature maps from shrinking after every convolution.

But padding is not new evidence. Zero padding adds artificial border values.

The model can learn to handle them, but they are not real pixels. That is the tradeoff:

```text
padding preserves spatial dimensions and border participation
padding also introduces artificial boundary assumptions
```

In [3]:
X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
]
)

padded = torch.zeros(4, 4)
padded[1:3, 1:3] = X # For rows and columns whoms' indices are between 1 and 2, make those values equal to X

print(padded)

# 0, 0, 0, 0
# 0, 1, 2, 0
# 0, 3, 4, 0
# 0, 0, 0, 0

conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
Y = conv(torch.zeros(1, 1, 5, 5)) # For kernel_size = 3 and padding = 1, 5+2*(1)-3+1 = 5

assert shape(padded) == (4, 4)
assert shape(Y) == (1, 1, 5, 5) # Padding preserves the input's spatial dimensions by adding a border around the input

tensor([[0., 0., 0., 0.],
        [0., 1., 2., 0.],
        [0., 3., 4., 0.],
        [0., 0., 0., 0.]])


Since we now need to account for padding, the formula for Y's heights and weights are:

$$
\text{output} =
\frac{\text{input} + 2(\text{padding}) - \text{kernel}}{\text{stride}} + 1
$$

Here:

- `input = 5`
- `kernel = 3`
- `padding = 1`
- `stride = 1`

Therefore:

$$
\text{output} =
\frac{5 + 2(1) - 3}{1} + 1
= 5
$$

So:

`Y.shape = (1, 1, 5, 5)`

# 7.3.3 Stride Skips Window Positions

Stride controls sampling density.

* Stride 1 asks the local detector at every possible position.
* Stride 2 asks it at every other position.
* Larger stride reduces the spatial size of the feature map and reduces computation, but it also discards some location detail.

The theoretical meaning is downsampling:

```text
smaller spatial map
larger effective jump between neighboring output values
less memory and compute
less precise spatial information
```

Stride is therefore an architectural decision. It affects what information later layers can access.

## Why use stride > 1?

- **Downsampling:** Creates a coarser spatial representation by skipping some windows.
- **Translation tolerance:** Makes the network less sensitive to small changes in exact feature location.
- **Larger receptive fields:** Allows deeper layers to incorporate information from larger regions of the original input.
- **Hierarchical features:** Helps CNNs move from fine details (edges/textures) toward higher-level features (shapes/objects).
- **Learned downsampling:** A strided convolution can learn which information is useful to preserve while reducing spatial resolution.

**Tradeoff:** Larger strides lose spatial precision, which is often acceptable for classification but more problematic for tasks requiring precise locations, such as segmentation.

In [ ]:
positions_stride_1 = list(range(0, 6 - 3 + 1, 1)) # Between 0 to 4, at the intervals of 1 (so 0, 1, 2, 3)
positions_stride_2 = list(range(0, 6 - 3 + 1, 2)) # Between 0 to 4, at the intervals of 2 (so 0, 2)

print("stride 1 starts:", positions_stride_1)
print("stride 2 starts:", positions_stride_2)

conv = nn.Conv2d(1, 1, kernel_size=3, stride=2) # Weight shape of (1, 1, 3, 3)
Y = conv(torch.zeros(1, 1, 6, 6)) # output = floor((6 - 3) / 2) + 1 = floor(1.5) + 1 = 2, giving shape (1, 1, 2, 2)

assert positions_stride_1 == [0, 1, 2, 3]
assert positions_stride_2 == [0, 2]
assert shape(Y) == (1, 1, 2, 2)

# 7.3.4 Padding and Stride Do Not Change Parameter Count

This is a key separation:

```text
parameter count: what weights are learned
output shape: where and how often those weights are applied
```

* Padding and stride change the spatial geometry of the computation, but they do not create new kernel weights.
* A 3 by 3 kernel over 3 input channels with 8 output channels has the same learned weight count regardless of whether it scans densely, skips positions, or uses padding.

This helps separate capacity from resolution.
* More channels or larger kernels increase parameter count.
* **Stride and padding mainly change feature-map size and border behavior**.

In [4]:
a = nn.Conv2d(3, 8, kernel_size=3, padding=0, stride=1)
b = nn.Conv2d(3, 8, kernel_size=3, padding=1, stride=2)

params_a = sum(p.numel() for p in a.parameters())
params_b = sum(p.numel() for p in b.parameters())

print("params without padding/stride:", params_a)
print("params with padding/stride:", params_b)

assert params_a == params_b == 8 * 3 * 3 * 3 + 8 # Same parameter count because both layers have the same input (3)/output (8) channels and kernel size (3*3); padding and stride do not add parameters

params without padding/stride: 224
params with padding/stride: 224


# 7.3.5 Break It Deliberately: Kernel Too Large

The kernel must fit somewhere. If the input is 3 by 3 and the kernel is 5 by 5 with no padding, there is no valid local window.

* The theory-level mistake is asking a local detector to inspect a neighborhood larger than the available representation.
* Padding could create artificial space around the input, but without padding, the operation has no valid output location.

This failure is useful because it connects the shape formula to a physical sliding-window interpretation.

In [5]:
conv = nn.Conv2d(1, 1, kernel_size=5)
X = torch.zeros(1, 1, 3, 3)

try:
    conv(X) # Doesn't work because the 5×5 kernel cannot fit inside the 3×3 input; the kernel must be smaller than or equal to the input's spatial dimensions
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The kernel should not fit inside the input.")

RuntimeError
Calculated padded input size per channel: (3 x 3). Kernel size: (5 x 5). Kernel size can't be greater than actual input size


#7.3 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. Why are padding and stride architectural choices rather than only API arguments?
> Padding and stride affect the spatial dimensions and resolution of feature maps, so they influence how information flows through the entire CNN architecture

2. What do padding and stride each control?
> * Padding controls whether the input will receive borders around the input (and if so by how big), and is deliberately used to maintain spatial fidelity between inputs and outputs in CNN. By default, CNN does not provide any padding (so output dimensions are smaller relative to input)
> * Stride controls the increments at which the kernel moves through the input. By default, `stride=1`, or moves 1 element at a time between kernel windows. Larger strides skip some possible windows and reduce spatial resolution

3. Why does `padding=1` preserve size for a 3 by 3 kernel with stride 1?
> * Padding of 1 adds one layer of values around each side of the input, effectively increasing each spatial dimension by 2
> * For a 3×3 kernel with stride 1, this exactly offsets the 3×3 kernel's reduction, so the output has the same spatial dimensions as the input

4. What information tradeoff does larger stride create?
> Larger stride reduces spatial precision by skipping some windows, trading detailed spatial information for a coarser representation

5. Why does stride affect output shape but not parameter count?
> Stride changes how many positions the kernel is applied to, so it changes the output shape. However, it does not change the kernel's weights, so it does not change the parameter count

6. What does it mean mechanically when the kernel is too large for the input?
> The kernel cannot fit completely inside the input, so there is no valid position where the convolution can be performed. PyTorch therefore raises a `RuntimeError`